#### Venv
Create a venv using python -m venv venv
activate the venv
and install the requirements.txt

### importing  required Libs

In [ ]:
import torch
import numpy as np
from datasets import Dataset, ClassLabel
from transformers import (
    XLMRobertaTokenizer,
    XLMRobertaForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, f1_score

### Loading the dataset

In [56]:
from datasets import load_dataset
dataset = load_dataset(
"csv",
data_files="trainV2.csv"
)


In [57]:
dataset["train"][:5]

{'Text': ['நான் கூட உன்னை வெகுளியான பொண்ணு&#39;னு நெனச்சிட்டேன் திவ்யா. நீ தெளிவாதான் இருக்க. உங்க வீடியோ பாக்குரேன்ல நான்தான் பைத்தியமா இருந்துருக்கேன்',
  'உன் போட்டோவை டாய்லெட்டுக்கு மாற்றினார்கள் அசிங்கமா தாண்டி இருக்கும்',
  'கண்டா வரச்சொல்லுங்க கார்த்திய திவ்யாவோட சேர்த்து வையுங்கள்',
  'ஒன்னோட சைசுக்கு நீயே ஒரு நாளக்கி 5கிலோ ஆய் போவ ஒனக்கு இந்த சைஸ் போதுமா?',
  'ரெண்டும் மிக பெரிய வெடிகுண்டு இவங்கள எதுக்கு ஷோ நடத்தி டெவலப் பண்றீயா'],
 'Class': ['Non-Abusive', 'Abusive', 'Non-Abusive', 'Abusive', 'Non-Abusive']}

Normalizing the data by
> Removing white space at end 
> Converting the case 

In [58]:
label2id = {
    "Non-Abusive": 0,
    "Abusive": 1
}
def encode_labels(example):
    label = example["Class"].strip()
    if label.lower() == "abusive":
        example["Class"] = label2id["Abusive"]
    elif label.lower() == "non-abusive":
        example["Class"] = label2id["Non-Abusive"]
    else:
        # Fallback: try direct lookup
        example["Class"] = label2id[label]
    return example
dataset = dataset.map(encode_labels)

In [59]:
dataset = dataset.rename_column("Text", "text")
dataset = dataset.rename_column("Class", "label")

In [60]:

def convert_label(example):
    example["label"] = int(example["label"])
    return example

dataset = dataset.map(convert_label)

In [61]:
dataset["train"][:5]

{'text': ['நான் கூட உன்னை வெகுளியான பொண்ணு&#39;னு நெனச்சிட்டேன் திவ்யா. நீ தெளிவாதான் இருக்க. உங்க வீடியோ பாக்குரேன்ல நான்தான் பைத்தியமா இருந்துருக்கேன்',
  'உன் போட்டோவை டாய்லெட்டுக்கு மாற்றினார்கள் அசிங்கமா தாண்டி இருக்கும்',
  'கண்டா வரச்சொல்லுங்க கார்த்திய திவ்யாவோட சேர்த்து வையுங்கள்',
  'ஒன்னோட சைசுக்கு நீயே ஒரு நாளக்கி 5கிலோ ஆய் போவ ஒனக்கு இந்த சைஸ் போதுமா?',
  'ரெண்டும் மிக பெரிய வெடிகுண்டு இவங்கள எதுக்கு ஷோ நடத்தி டெவலப் பண்றீயா'],
 'label': ['0', '1', '0', '1', '0']}

In [77]:

from datasets import ClassLabel
dataset = dataset.cast_column("label", ClassLabel(num_classes=2, names=["Non-Abusive", "Abusive"]))

dataset_split = dataset["train"].train_test_split(
    test_size=0.2,
    seed=42,
    stratify_by_column="label"
)
train_dataset = dataset_split["train"]
test_dataset = dataset_split["test"]

Casting the dataset: 100%|██████████| 3652/3652 [00:00<00:00, 456682.81 examples/s]


In [74]:
train_dataset[:5]

{'text': ['நீங்க நல்ல குடும்பம் தானே சார் ஏ ன் நீங்க நடிக்க அனுபுறிங்க. உங்க பொன்ன உங்க கேரளா பையனா பாத்து கல்யாணம் பண்ணி வச்சு இருக்கலாமே.',
  'நல்லா சிந்தித்து பாருங்கள் அன்று பெண்கள் சாதாரண மொபைல் போன் இருக்கும் பொது பெண்கள் நல்லா இருந்தாங்க இன்னைக்கு டிக் டாக் பெண்கள் வாழ்கைய தழை கீலா போட்றிச்சி இதுதான் பெண்கள் சுதந்திரமா இதலாம் பாக்கனும்னு தழைஎலுத்து',
  'அவுத்துபோட்டு ஆடுறது எல்லாம் முதலமைச்சர் ஜெயலலிதா ஆகி விட முடியாது,மரியாதையா குடும்பத்துடன் இரு ரசனிதாத்தா மாதிரி தான் உனக்கும் பிஜேபி கிட்ட ஆப்பு ரெடி யா இருக்கு',
  'நாட்டுல்ல அவ்வளவு பிரச்சினை இருக்கும் போது நீங்க எதுக்கு இந்த உதவாகறைங்கல கூப்டு வச்சி எங்க நேரத்தையும் வீணாக்கி உங்க ட்ரெண்டிங் ஏத்திக்கிறிங்க...',
  'உங்களை மாறி சேனல் தாண்டா, உங்க வியூஸ்காக இந்த கழிசடை எல்லா கூட்டிட்டு வந்து இன்டர்வியூ எடுக்கறான்னு எல்லாத்தையும் அவல பேச வி\u200cட்டு தெரிய வைக்கிறீங்க,'],
 'label': [1, 0, 1, 0, 0]}

In [78]:
test_dataset[:5]

{'text': ['இதுக்கு பதிலா இவ தூகிட்டு சாகலாம். த்து.',
  'சீ நீ என்ன காதல் சைக்கோவா... ஒரு சமுதாயத்தையே கலங்க வக்கர.... காதலையே கொச்சை படுத்தற... உன் பிரச்சனைகளை இவ்ளோ தூரம் பெரிய பெரியவங்ககிட்ட கொண்டு போற அவங்க டைமும் வேஸ்ட் இதை பாக்கற எங்க எல்லாத்துக்கும் டைம் வேஸ்ட்... நாட்டுல இருக்கற கொரோனா பிரச்சனைல பாதிக்கப்பட்டு இன்னும் மீண்டு வராம நெறைய குடும்பங்கள் இருக்கு... மரணத்தோட போராடுறாங்க, ஒரு வேளை சாப்பாட்டுக்கே வழி இல்லாம கூட நெறைய ஜனங்க தமிழ்நாட்டுல கஷ்டப்பட்றாங்க... இதுல உன் பிரச்சனை ஒரு பிரச்சனையா சொல்லு',
  'கார்த்தி வந்து ரொணடு பொத்தையும் நல்லா குத்து எம்புல் டாணக்கதா மருந்து',
  'உன்ன மட்டும் இன்டெர்வியூ எடுக்குற வாய்ப்பு எனக்கு கிடைச்சா நல்லா 4 அரை விட்டு நறுக்குன்னு அட்வைஸ் பண்ணி அனுப்பி வைப்பேன்... படிச்ச முட்டாள்ன்னு சொல்லுவாங்க.. அது உனக்கு 100% பொருந்தும்... உங்க அம்மாவை நினைச்சா தான் பாவமா இருக்கு...',
  'வற வற தோப்பு ரொம்ப பெருசாகிட்டே போகுதே...தண்ணி ரொம்ப பாய்தோ....'],
 'label': [1, 0, 0, 0, 1]}